In [1]:
import os
from typing import Literal
from dotenv import load_dotenv
from anthropic import Anthropic
from openai import OpenAI

Provider = Literal["anthropic", "openai"]
load_dotenv()

def get_client(provider: Provider):
    if provider == "anthropic":
        api_key = os.environ.get("ANTHROPIC_API_KEY")
        if not api_key:
            raise ValueError("Missing ANTHROPIC_API_KEY")
        return Anthropic(api_key=api_key)

    if provider == "openai":
        api_key = os.environ.get("OPENAI_API_KEY")
        if not api_key:
            raise ValueError("Missing OPENAI_API_KEY")
        return OpenAI(api_key=api_key)

    raise ValueError(f"Unsupported provider: {provider}")


def get_default_model(provider: Provider) -> str:
    if provider == "anthropic":
        return "claude-sonnet-4-6"
    if provider == "openai":
        return "gpt-5.4"
    raise ValueError(f"Unsupported provider: {provider}")


def extract_anthropic_text(response) -> str:
    parts = []
    for block in response.content:
        text = getattr(block, "text", None)
        if text:
            parts.append(text)
    return "\n".join(parts).strip()


def extract_openai_text(response) -> str:
    if hasattr(response, "output_text") and response.output_text:
        return response.output_text.strip()

    parts = []
    for item in getattr(response, "output", []):
        for content in getattr(item, "content", []):
            text = getattr(content, "text", None)
            if text:
                parts.append(text)
    return "\n".join(parts).strip()


def ask_model(
    prompt: str,
    provider: Provider,
    model: str | None = None,
    max_tokens: int = 400,
) -> str:
    client = get_client(provider)
    model = model or get_default_model(provider)

    if provider == "anthropic":
        response = client.messages.create(
            model=model,
            max_tokens=max_tokens,
            messages=[{"role": "user", "content": prompt}],
        )
        return extract_anthropic_text(response)

    if provider == "openai":
        response = client.responses.create(
            model=model,
            input=prompt,
            max_output_tokens=max_tokens,
        )
        return extract_openai_text(response)

    raise ValueError(f"Unsupported provider: {provider}")

In [3]:
import os
from dotenv import load_dotenv

load_dotenv()

print("ANTHROPIC:", bool(os.getenv("ANTHROPIC_API_KEY")))
print("OPENAI:", bool(os.getenv("OPENAI_API_KEY")))
print("cwd:", os.getcwd())

ANTHROPIC: True
OPENAI: True
cwd: /workspaces/agent_experimental_builds/01-foundation


In [5]:
print(ask_model("Say hello from Claude.", provider="anthropic"))

Hello! 👋 I'm Claude, an AI assistant made by Anthropic. It's great to meet you! How can I help you today?


In [ ]:
print(ask_model("Say hello from GPT.", provider="openai"))

In [7]:
#test minsearch
from minsearch import Index

docs = [
    {
        "question": "How do I join the course after it has started?",
        "text": "You can join the course at any time. We have recordings available.",
        "section": "General Information",
        "course": "data-engineering-zoomcamp"
    },
    {
        "question": "What are the prerequisites for the course?",
        "text": "You need to have basic knowledge of programming.",
        "section": "Course Requirements",
        "course": "data-engineering-zoomcamp"
    }
]

index = Index(
    text_fields=["question", "text", "section"],
    keyword_fields=["course"]
)
index.fit(docs)

query = "Can I join the course if it has already started?"
filter_dict = {"course": "data-engineering-zoomcamp"}
boost_dict = {"question": 3, "text": 1, "section": 1}

results = index.search(query, filter_dict=filter_dict, boost_dict=boost_dict)
results

[{'question': 'How do I join the course after it has started?',
  'text': 'You can join the course at any time. We have recordings available.',
  'section': 'General Information',
  'course': 'data-engineering-zoomcamp'},
 {'question': 'What are the prerequisites for the course?',
  'text': 'You need to have basic knowledge of programming.',
  'section': 'Course Requirements',
  'course': 'data-engineering-zoomcamp'}]

In [8]:
from minsearch import VectorSearch
import numpy as np

vectors = np.random.rand(3, 768)
payload = [
    {"id": 1, "title": "Python Tutorial", "category": "programming", "level": "beginner"},
    {"id": 2, "title": "Data Science Guide", "category": "data", "level": "intermediate"},
    {"id": 3, "title": "Machine Learning Basics", "category": "ai", "level": "advanced"},
]

index = VectorSearch(keyword_fields=["category", "level"])
index.fit(vectors, payload)

query_vector = np.random.rand(768)
filter_dict = {"category": "programming", "level": "beginner"}

results = index.search(query_vector, filter_dict=filter_dict, num_results=5)
results

[{'id': 1,
  'title': 'Python Tutorial',
  'category': 'programming',
  'level': 'beginner'}]

In [ ]:
# This is Cool. 
from minsearch import VectorSearch
import numpy as np

# Create the index
index = VectorSearch(keyword_fields=["category", "level"])

# Append a single vector
vector = np.random.rand(768)
doc = {"id": 1, "title": "Python Tutorial", "category": "programming", "level": "beginner"}
index.append(vector, doc)

# Append multiple vectors in batch
vectors = np.random.rand(10, 768)
payload = [
    {"id": i+2, "title": f"Document {i+2}", "category": "data", "level": "intermediate"}
    for i in range(10)
]
index.append_batch(vectors, payload)

# Search works the same way
query_vector = np.random.rand(768)
results = index.search(query_vector, num_results=5)

In [ ]:

# Working when bedrock access was clear
import os
from anthropic import Anthropic, AnthropicBedrock

direct_client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
bedrock_client = AnthropicBedrock(aws_region=os.environ.get("AWS_REGION", "us-east-1"))

def claude_client(provider: str = "direct"):
    provider = provider.lower().strip()
    if provider == "direct":
        return direct_client, "claude-sonnet-4-5"
    if provider == "bedrock":
        return bedrock_client, "arn:aws:bedrock:us-east-1:053920690235:inference-profile/global.anthropic.claude-sonnet-4-20250514-v1:0"
    raise ValueError("provider must be 'direct' or 'bedrock'")

def ask_claude(prompt: str, provider: str = "direct", max_tokens: int = 400):
    client, model = claude_client(provider)
    return client.messages.create(
        model=model,
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}],
    )

direct_resp = ask_claude("Say hello from direct Claude.", provider="direct")
bedrock_resp = ask_claude("Say hello from Bedrock Claude.", provider="bedrock")

print("DIRECT:")
print(direct_resp.content)

print("\nBEDROCK:")
print(bedrock_resp.content)

PermissionDeniedError: Error code: 403 - {'message': 'The security token included in the request is invalid.'}